# RAGDoll graded-results dashboard

Visualizes `graded.jsonl` and the CSV files under `scores/`. Run all cells after reopening the notebook; paths are resolved whether Jupyter starts in this folder or at the repository root.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

def find_results_dir():
    cwd = Path.cwd().resolve()
    relative = Path("evaluation-results/aus-agent-pilot/rubric/judged")
    candidates = [cwd, cwd / relative]
    candidates += [parent / relative for parent in cwd.parents]
    for candidate in candidates:
        if (candidate / "graded.jsonl").exists():
            return candidate
    raise FileNotFoundError("Could not find judged/graded.jsonl from " + str(cwd))

RESULTS_DIR = find_results_dir()
SCORES_DIR = RESULTS_DIR / "scores"
print(f"Loading results from: {RESULTS_DIR}")

In [ ]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

graded_rows = read_jsonl(RESULTS_DIR / "graded.jsonl")
graded = pd.DataFrame(graded_rows)
cell_scores = pd.read_csv(SCORES_DIR / "cell_scores.csv")
run_scores = pd.read_csv(SCORES_DIR / "run_scores.csv")
category_failures = pd.read_csv(SCORES_DIR / "category_failures.csv")

criterion_rows = []
for row in graded_rows:
    for index, criterion in enumerate(row.get("criteria", []), start=1):
        criterion_rows.append({
            "qid": str(row.get("qid", "")),
            "run_id": row.get("run_id", ""),
            "criterion_number": index,
            "criterion": criterion.get("text", criterion.get("criterion", "")),
            "axis": criterion.get("axis", criterion.get("type", "unknown")),
            "type": criterion.get("type", "unknown"),
            "weight": criterion.get("weight", 0),
            "verdict": criterion.get("verdict", "missing"),
        })
criteria = pd.DataFrame(criterion_rows)

summary = pd.DataFrame({
    "metric": ["graded cells", "runs", "topics", "criteria", "failed cells"],
    "value": [len(graded), graded.run_id.nunique(), graded.qid.nunique(), len(criteria),
              int((graded.status != "completed").sum())],
})
display(summary.style.hide(axis="index"))
display(run_scores.sort_values("ternary_score", ascending=False).style.format({
    "ternary_score": "{:.3f}", "binary_score": "{:.3f}"
}))

## Overall run comparison

In [ ]:
plot_runs = run_scores.sort_values("ternary_score")
ax = plot_runs.plot.barh(
    x="run_id", y=["ternary_score", "binary_score"],
    figsize=(10, max(4, len(plot_runs) * 0.65)), color=["#2563eb", "#93c5fd"]
)
ax.set(xlabel="Score", ylabel="", xlim=(0, 1), title="Mean score by run")
ax.legend(["Ternary (partial credit)", "Binary"])
plt.tight_layout()
plt.show()

## Run × topic heatmap

In [ ]:
heat = cell_scores.pivot(index="run_id", columns="qid", values="ternary_score")
fig, ax = plt.subplots(figsize=(max(8, heat.shape[1] * 1.5), max(4, heat.shape[0] * 0.7)))
image = ax.imshow(heat, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(heat.shape[1]), labels=heat.columns, rotation=40, ha="right")
ax.set_yticks(range(heat.shape[0]), labels=heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Ternary score by run and topic")
fig.colorbar(image, ax=ax, label="Score")
plt.tight_layout()
plt.show()

## Verdicts and rubric axes

Negative-weight criteria are penalties. Their verdicts describe whether the condition written in the criterion was judged present; use the final weighted scores for overall ranking.

In [ ]:
verdict_order = ["satisfied", "partially_satisfied", "not_satisfied", "failed", "missing"]
verdict_counts = pd.crosstab(criteria["run_id"], criteria["verdict"]).reindex(columns=verdict_order, fill_value=0)
verdict_rates = verdict_counts.div(verdict_counts.sum(axis=1), axis=0)
ax = verdict_rates.plot.bar(
    stacked=True, figsize=(11, 5),
    color=["#16a34a", "#facc15", "#dc2626", "#7f1d1d", "#94a3b8"]
)
ax.set(xlabel="", ylabel="Share of criteria", title="Criterion verdict mix by run")
ax.legend(title="Verdict", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

axis_rates = (
    criteria.assign(satisfied=criteria.verdict.map({"satisfied": 1.0, "partially_satisfied": 0.5}).fillna(0.0))
    .groupby(["axis", "run_id"], as_index=False)["satisfied"].mean()
    .pivot(index="axis", columns="run_id", values="satisfied")
)
display(axis_rates.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

## Criterion failure categories

In [ ]:
display(category_failures.sort_values("failure_rate", ascending=False).style.format({"failure_rate": "{:.1%}"}))
ax = category_failures.sort_values("failure_rate").plot.barh(
    x="type", y="failure_rate", legend=False, figsize=(9, 5), color="#f97316"
)
ax.set(xlabel="Failure rate", ylabel="", xlim=(0, 1), title="Failure rate by criterion type")
plt.tight_layout()
plt.show()

## Per-query drill-down

Change `SELECTED_QID` and `SELECTED_RUN` below, then rerun the cell.

In [ ]:
SELECTED_QID = str(cell_scores.iloc[0]["qid"])
SELECTED_RUN = str(cell_scores.iloc[0]["run_id"])

matches = [r for r in graded_rows if str(r.get("qid")) == SELECTED_QID and str(r.get("run_id")) == SELECTED_RUN]
if not matches:
    print("No matching row. Available combinations:")
    display(cell_scores[["qid", "run_id"]])
else:
    row = matches[0]
    score_row = cell_scores[(cell_scores.qid.astype(str) == SELECTED_QID) & (cell_scores.run_id == SELECTED_RUN)]
    display(Markdown(f"### {SELECTED_RUN} — `{SELECTED_QID}`"))
    display(score_row.style.format({"ternary_score": "{:.3f}", "binary_score": "{:.3f}"}))
    display(Markdown("**Query**\n\n" + str(row.get("query", ""))))
    detail = criteria[(criteria.qid == SELECTED_QID) & (criteria.run_id == SELECTED_RUN)].copy()
    display(detail[["criterion_number", "axis", "weight", "verdict", "criterion"]].style.map(
        lambda value: "background-color: #fee2e2" if value in {"not_satisfied", "failed"} else
                      "background-color: #fef9c3" if value == "partially_satisfied" else
                      "background-color: #dcfce7" if value == "satisfied" else "",
        subset=["verdict"],
    ))
    display(Markdown("**Answer**\n\n" + str(row.get("answer_text", ""))))